**Part 2 · LangChain** · step 6 of 25  
[← 05 Messages](05_messages.ipynb) · [🏠 Roadmap](../README.md) · [07 Structured output →](07_structured_output.ipynb)

# 06 · Prompt Templates & LCEL Chains

**What you'll learn**
- `PromptTemplate` — a reusable string with `{placeholders}`.
- `ChatPromptTemplate` — the same idea for a *list of messages*.
- `MessagesPlaceholder` — slot an entire chat history into a prompt.
- **LCEL** (LangChain Expression Language): chain steps with `|` → `prompt | model | parser`.

In [2]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

True

## 1. `PromptTemplate` — single variable

In [3]:
from langchain_core.prompts import PromptTemplate

template = """
You are an AI Tutor.

Answer the following question.

Question :
{question}
"""

prompt = PromptTemplate.from_template(template)
result = prompt.invoke({"question": "What is kafka ??"})  # -> StringPromptValue
print(result)

text='\nYou are an AI Tutor.\n\nAnswer the following question.\n\nQuestion :\nWhat is kafka ??\n'


In [4]:
# .to_string() gives the final text that would be sent to the model
print(result.to_string())


You are an AI Tutor.

Answer the following question.

Question :
What is kafka ??



## 2. Multiple variables

In [5]:
prompt = PromptTemplate.from_template(
    """
You are a {role}.

Explain me {topic}
in {level} level.
"""
)

prompt.invoke({"role": "Teacher", "topic": "Kafka", "level": "Hard"}).to_string()

'\nYou are a Teacher.\n\nExplain me Kafka\nin Hard level.\n'

## 3. `ChatPromptTemplate` + LCEL chain
`prompt | model | parser` creates a **chain**. Calling `chain.invoke(dict)` runs each step and pipes
the output of one into the next:

`dict` → **prompt** → messages → **model** → `AIMessage` → **StrOutputParser** → `str`

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}"),
])
model = init_chat_model("gpt-4o-mini")
parser = StrOutputParser()  # AIMessage -> plain string

chain = prompt | model | parser

result = chain.invoke({"question": "Explain Kafka partitions."})
print(result)

Kafka partitions are a key feature of Apache Kafka, a distributed messaging system designed to handle high-throughput, real-time data feeds. Partitions allow Kafka to scale both in terms of data and consumer load. Here's a detailed explanation of Kafka partitions:

### What is a Partition?

1. **Definition**:
   - A partition is a single ordered and immutable sequence of messages within a Kafka topic. Each message is assigned a unique offset (a sequential identifier) within its partition.

2. **Structure**:
   - Each partition is a log that stores messages in the order they arrive. The messages in a partition are accessible via their offsets, which start from zero and increment with each new message.

### Key Features of Partitions:

1. **Scalability**:
   - Multiple partitions within a topic allow Kafka to scale horizontally. By adding more partitions, you can distribute the message load more evenly across multiple brokers in a Kafka cluster.

2. **Parallel Processing**:
   - Consumer

## 4. `MessagesPlaceholder` — inject chat history
Reserves a spot where a whole **list of messages** goes. This is how you add memory to a chain.

In [7]:
from langchain_core.prompts import MessagesPlaceholder
from langchain.messages import HumanMessage, AIMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

result = prompt.invoke({
    "history": [HumanMessage(content="Hi"), AIMessage(content="Hello")],
    "question": "Explain Kafka",
})
print(result.to_string())

System: You are helpful.
Human: Hi
AI: Hello
Human: Explain Kafka


## Revision notes
| Tool | Input | Output |
|---|---|---|
| `PromptTemplate` | `dict` | `StringPromptValue` (`.to_string()`) |
| `ChatPromptTemplate` | `dict` | `ChatPromptValue` (`.to_messages()`) |
| `MessagesPlaceholder("history")` | list of messages | expands in place |
| `StrOutputParser` | `AIMessage` | `str` |
| `a \| b \| c` | — | a chain: output of `a` becomes input of `b` |

---
✅ **Done with 06 Prompt templates & chains.** Next: [07 Structured output →](07_structured_output.ipynb)  
[🏠 Roadmap](../README.md)